# Fuzzy logika a fuzzy modelování
Tento notebook obsahuje řešení dvou úloh z oblasti fuzzy logiky podle předepsaných kroků pomocí knihovny `scikit-fuzzy`.

*Než začnete, ujistěte se, že máte nainstalované potřebné knihovny: `pip install scikit-fuzzy matplotlib numpy`*

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

## První úloha: Klimatizační systém

### 1. Definice fuzzy proměnných
- **Vstupní proměnné:** teplota v místnosti a cílová teplota (rozsah 0–40 °C).
- **Výstupní proměnná:** příkaz pro klimatizační systém (rozsah -5 až 5).
- Použití **Gaussových** příslušnostních funkcí pro všechny proměnné.

In [ ]:
# Definice univerza (rozsahů)
room_temp = ctrl.Antecedent(np.arange(0, 41, 1), 'room_temp')
target_temp = ctrl.Antecedent(np.arange(0, 41, 1), 'target_temp')
command = ctrl.Consequent(np.arange(-5, 6, 1), 'command')

# Gaussovy příslušnostní funkce pro vstupy (centra 0, 10, 20, 30, 40)
terms = ['Very Cold', 'Cold', 'Warm', 'Hot', 'Very Hot']
centers = [0, 10, 20, 30, 40]

for term, center in zip(terms, centers):
    room_temp[term] = fuzz.gaussmf(room_temp.universe, center, 4)
    target_temp[term] = fuzz.gaussmf(target_temp.universe, center, 4)

# Gaussovy příslušnostní funkce pro výstup
command['Cool'] = fuzz.gaussmf(command.universe, -5, 1.5)
command['No Change'] = fuzz.gaussmf(command.universe, 0, 1.5)
command['Heat'] = fuzz.gaussmf(command.universe, 5, 1.5)

# Zobrazení funkcí příslušnosti (odkomentujte pro zobrazení)
# room_temp.view()
# command.view()

### 2. Definice pravidel fuzzy logiky a 3. Implementace modelu
Matice pravidel je vytvořena ze zadání úlohy. Následně sestavíme inferenční řídicí systém.

In [ ]:
rules = [
    # Room Temp: Very Cold
    ctrl.Rule(room_temp['Very Cold'] & target_temp['Very Cold'], command['No Change']),
    ctrl.Rule(room_temp['Very Cold'] & target_temp['Cold'], command['Heat']),
    ctrl.Rule(room_temp['Very Cold'] & target_temp['Warm'], command['Heat']),
    ctrl.Rule(room_temp['Very Cold'] & target_temp['Hot'], command['Heat']),
    ctrl.Rule(room_temp['Very Cold'] & target_temp['Very Hot'], command['Heat']),
    
    # Room Temp: Cold
    ctrl.Rule(room_temp['Cold'] & target_temp['Very Cold'], command['Cool']),
    ctrl.Rule(room_temp['Cold'] & target_temp['Cold'], command['No Change']),
    ctrl.Rule(room_temp['Cold'] & target_temp['Warm'], command['Heat']),
    ctrl.Rule(room_temp['Cold'] & target_temp['Hot'], command['Heat']),
    ctrl.Rule(room_temp['Cold'] & target_temp['Very Hot'], command['Heat']),
    
    # Room Temp: Warm
    ctrl.Rule(room_temp['Warm'] & target_temp['Very Cold'], command['Cool']),
    ctrl.Rule(room_temp['Warm'] & target_temp['Cold'], command['Cool']),
    ctrl.Rule(room_temp['Warm'] & target_temp['Warm'], command['No Change']),
    ctrl.Rule(room_temp['Warm'] & target_temp['Hot'], command['Heat']),
    ctrl.Rule(room_temp['Warm'] & target_temp['Very Hot'], command['Heat']),

    # Room Temp: Hot
    ctrl.Rule(room_temp['Hot'] & target_temp['Very Cold'], command['Cool']),
    ctrl.Rule(room_temp['Hot'] & target_temp['Cold'], command['Cool']),
    ctrl.Rule(room_temp['Hot'] & target_temp['Warm'], command['Cool']),
    ctrl.Rule(room_temp['Hot'] & target_temp['Hot'], command['No Change']),
    ctrl.Rule(room_temp['Hot'] & target_temp['Very Hot'], command['Heat']),

    # Room Temp: Very Hot
    ctrl.Rule(room_temp['Very Hot'] & target_temp['Very Cold'], command['Cool']),
    ctrl.Rule(room_temp['Very Hot'] & target_temp['Cold'], command['Cool']),
    ctrl.Rule(room_temp['Very Hot'] & target_temp['Warm'], command['Cool']),
    ctrl.Rule(room_temp['Very Hot'] & target_temp['Hot'], command['Cool']),
    ctrl.Rule(room_temp['Very Hot'] & target_temp['Very Hot'], command['No Change']),
]

# Implementace modelu pomocí scikit-fuzzy
ac_ctrl = ctrl.ControlSystem(rules)
ac_sim = ctrl.ControlSystemSimulation(ac_ctrl)

### 4. Testování modelu a 5. Vizualizace výsledků
- Test pro parametry: teplota v místnosti 30 °C a cílová 20 °C (Očekává se chlazení/Cool).
- 3D graf zobrazující závislost vstupů na výstupu.

In [ ]:
# Testování
ac_sim.input['room_temp'] = 30
ac_sim.input['target_temp'] = 20
ac_sim.compute()

print("---- Testování modelu ----")
print(f"Vstup: Místnost = 30°C, Cílová = 20°C")
print(f"Výstup (Command): {ac_sim.output['command']:.2f}")
print("Očekávaná hodnota: Zhruba -5 (Cool)\n")

# 3D Vizualizace
x, y = np.meshgrid(np.arange(0, 41, 1), np.arange(0, 41, 1))
z = np.zeros_like(x, dtype=float)

for i in range(41):
    for j in range(41):
        ac_sim.input['room_temp'] = x[i, j]
        ac_sim.input['target_temp'] = y[i, j]
        ac_sim.compute()
        z[i, j] = ac_sim.output['command']

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(x, y, z, cmap='coolwarm', edgecolor='none')
ax.set_xlabel('Teplota v místnosti (°C)')
ax.set_ylabel('Cílová teplota (°C)')
ax.set_zlabel('Příkaz (-5 = Cool, 5 = Heat)')
plt.title('3D závislost příkazu na teplotách')
plt.show()

---
## Druhá úloha: Fuzzy model pro predikci výsledku zkoušky

### Analýza vstupů
Na základě tabulky studentů vyvodíme následující závěry:
1. **Doba studia:** Zásadní faktor. Pokud student studoval "Více než 4 dny", zkoušku zpravidla složil. Student, který nestudoval vůbec, zkoušku nesložil bez ohledu na výbornou známku (Rudolf J.).
2. **Průměrná známka:** Působí jako hraniční faktor. Pokud student studoval "Méně než 4 dny", výsledek závisel na známce (např. průměrná známka do ~3.0 stačila na složení, známka >3.0 znamenala neúspěch).
3. Populárnost a IQ jsou spíše okrajové (distrakční) znaky.

In [ ]:
# Převod vybraných atributů do fuzzy podoby (Doba studia a Známka)
study_time = ctrl.Antecedent(np.arange(0, 11, 1), 'study_time') # 0-10 dní
grade = ctrl.Antecedent(np.arange(1.0, 5.1, 0.1), 'grade') # Známky jako na ZŠ/SŠ (1.0 - 5.0)
pass_chance = ctrl.Consequent(np.arange(0, 101, 1), 'pass_chance') # Šance v procentech

# Fuzzy množiny
study_time['Malo_Zadne'] = fuzz.trapmf(study_time.universe, [0, 0, 1, 3])
study_time['Kratce'] = fuzz.trimf(study_time.universe, [2, 3, 5])
study_time['Dlouho'] = fuzz.trapmf(study_time.universe, [4, 5, 10, 10])

grade['Vyborna'] = fuzz.trimf(grade.universe, [1.0, 1.0, 2.5])
grade['Prumerna'] = fuzz.trimf(grade.universe, [2.0, 3.0, 4.0])
grade['Spatna'] = fuzz.trimf(grade.universe, [3.5, 5.0, 5.0])

pass_chance['Neslozil'] = fuzz.trimf(pass_chance.universe, [0, 0, 50])
pass_chance['Slozil'] = fuzz.trimf(pass_chance.universe, [50, 100, 100])

# Sestavení pravidel IF-THEN
exam_rules = [
    ctrl.Rule(study_time['Malo_Zadne'], pass_chance['Neslozil']), # Bez studia to nejde
    ctrl.Rule(study_time['Dlouho'], pass_chance['Slozil']),       # Pilné studium znamená úspěch
    ctrl.Rule(study_time['Kratce'] & grade['Vyborna'], pass_chance['Slozil']), 
    ctrl.Rule(study_time['Kratce'] & grade['Prumerna'], pass_chance['Slozil']), # Pokud se studovalo krátce, dobrý průměr zachraňuje
    ctrl.Rule(study_time['Kratce'] & grade['Spatna'], pass_chance['Neslozil'])  # Špatný průměr + krátké studium = neúspěch
]

# Implementace modelu
exam_ctrl = ctrl.ControlSystem(exam_rules)
exam_sim = ctrl.ControlSystemSimulation(exam_ctrl)

### Otestování chování modelu
Nasimulujeme příklady z tabulky: 
- Ladislav G. (Méně než 4 dny, známka 3.1) -> Očekáváme "Ne"
- Anna K. (Více než 4 dny, známka 1.3) -> Očekáváme "Ano"
- Rudolf J. (Nestudoval, známka 1.4) -> Očekáváme "Ne"

In [ ]:
test_students = [
    {"name": "Ladislav G.", "time": 3, "grade": 3.1},
    {"name": "Anna K.", "time": 6, "grade": 1.3},
    {"name": "Rudolf J.", "time": 0, "grade": 1.4}
]

print("---- Predikce Výsledku Zkoušky ----")
for student in test_students:
    exam_sim.input['study_time'] = student['time']
    exam_sim.input['grade'] = student['grade']
    exam_sim.compute()
    
    chance = exam_sim.output['pass_chance']
    result = "Ano (Složil)" if chance >= 50 else "Ne (Nesložil)"
    print(f"Student: {student['name']:<12} | Šance: {chance:>5.1f}% | Výsledek: {result}")